**1. Tratar valores nulos segun su mecanismo (MCAR, MAR,MNAR):**

*   Imputacion por media/mediana/moda.
*   KNNImputer, IterativeImputer

*Separar tipos de variables*

In [ ]:
import pandas as pd

# Cargar datos
df = pd.read_csv("/content/06-kickAutomotriz.csv")

# Numéricas
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Categóricas
cat_cols = df.select_dtypes(include=['object']).columns

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

*`Convertir fecha correctamente`*

In [ ]:
df['PurchDate'] = pd.to_datetime(df['PurchDate'])

Re-definir variables correctamente

In [ ]:
# Variables numéricas reales
num_cols = [
    'VehYear', 'VehicleAge', 'VehOdo',
    'MMRAcquisitionAuctionAveragePrice',
    'MMRAcquisitionAuctionCleanPrice',
    'MMRAcquisitionRetailAveragePrice',
    'MMRAcquisitonRetailCleanPrice',
    'MMRCurrentAuctionAveragePrice',
    'MMRCurrentAuctionCleanPrice',
    'MMRCurrentRetailAveragePrice',
    'MMRCurrentRetailCleanPrice',
    'VehBCost',
    'WarrantyCost'
]

# Variables categóricas (incluyendo IDs)
cat_cols = [
    'Auction', 'Make', 'Model', 'Trim', 'SubModel',
    'Color', 'Transmission', 'WheelType',
    'Nationality', 'Size', 'TopThreeAmericanName',
    'PRIMEUNIT', 'AUCGUART', 'VNST',
    'BYRNO', 'VNZIP1', 'WheelTypeID'
]

*Imputación simple (MCAR)*

In [ ]:
from sklearn.impute import SimpleImputer

simple_cols = ['VehYear', 'VehicleAge', 'VehOdo', 'WarrantyCost']

imputer_median = SimpleImputer(strategy='median')
df[simple_cols] = imputer_median.fit_transform(df[simple_cols])

*Numéricas complejas (MAR → KNN)*

In [ ]:
from sklearn.impute import KNNImputer

knn_cols = [
    'MMRAcquisitionAuctionAveragePrice',
    'MMRAcquisitionAuctionCleanPrice',
    'MMRAcquisitionRetailAveragePrice',
    'MMRAcquisitonRetailCleanPrice',
    'MMRCurrentAuctionAveragePrice',
    'MMRCurrentAuctionCleanPrice',
    'MMRCurrentRetailAveragePrice',
    'MMRCurrentRetailCleanPrice',
    'VehBCost'
]

imputer_knn = KNNImputer(n_neighbors=5)
df[knn_cols] = imputer_knn.fit_transform(df[knn_cols])

*Categóricas (moda)*

In [ ]:
imputer_cat = SimpleImputer(strategy='most_frequent')
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])

*Convierte la fecha en variables útiles:*

In [ ]:
df['year'] = df['PurchDate'].dt.year
df['month'] = df['PurchDate'].dt.month

**2. Detectar y tratar outliers (IQR, z-score, winsorizing)**

*¿En qué variables buscar outliers?*

In [ ]:
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1

outliers_iqr = ((df[num_cols] < (Q1 - 1.5 * IQR)) |
                (df[num_cols] > (Q3 + 1.5 * IQR)))

print(outliers_iqr.sum())

Muchos outliers en:

*   MMRCurrentAuctionCleanPrice → 1334
*   MMRAcquisitonRetailCleanPrice → 1261
*   WarrantyCost → 838

*Aplicar clipping (winsorizing con IQR)*

In [ ]:
df_out = df.copy()

for col in num_cols:
    Q1 = df_out[col].quantile(0.25)
    Q3 = df_out[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    df_out[col] = df_out[col].clip(lower, upper)

In [ ]:
import matplotlib.pyplot as plt

plt.boxplot(df['VehBCost'])
plt.title("Antes")
plt.show()

plt.boxplot(df_out['VehBCost'])
plt.title("Después")
plt.show()

Detección correcta de outliers con IQR
 

*   No se eliminó datos (decisión correcta)
*   Se Aplicó winsorizing para mantener robustez del modelo

**3. Eliminar duplicados y registros inconsistentes**

In [ ]:
# Número de duplicados exactos
print("Duplicados totales:", df.duplicated().sum())

# Verlos
df[df.duplicated()]

In [ ]:
df.duplicated(subset=['Make', 'Model', 'VehYear', 'VehOdo']).sum()

In [ ]:
df[df.duplicated(subset=['Make', 'Model', 'VehYear', 'VehOdo'], keep=False)].head(10)

*Eliminación de duplicados:*

No se encontraron registros completamente duplicados.
Se detectaron 213 posibles duplicados basados en variables (Make, Model, VehYear, VehOdo).
Sin embargo, estos no fueron eliminados, ya que representan vehículos distintos con características similares.
Se decidió conservarlos para no perder información relevante.

In [ ]:
# Valores inválidos
print("VehOdo <= 0:", (df['VehOdo'] <= 0).sum())
print("VehBCost <= 0:", (df['VehBCost'] <= 0).sum())
print("VehYear raro:", ((df['VehYear'] < 1980) | (df['VehYear'] > 2026)).sum())

*   No hay valores imposibles
*   No hay errores evidentes
*   Dataset consistente en variables clave

In [ ]:
df['edad_calculada'] = 2026 - df['VehYear']

# Ver inconsistencias
(df['edad_calculada'] != df['VehicleAge']).sum()

In [ ]:
df['VehicleAge'] = df['PurchDate'].dt.year - df['VehYear']

In [ ]:
df['edad_calculada'] = df['PurchDate'].dt.year - df['VehYear']

(df['edad_calculada'] != df['VehicleAge']).sum()

*Consistencia de variables:*

Se detectaron inconsistencias significativas entre VehicleAge y VehYear.
Se identificó que VehicleAge no era confiable.
Se recalculó utilizando la diferencia entre el año de compra (PurchDate) y el año del vehículo (VehYear).
Posteriormente, se verificó que no existan inconsistencias (0 registros).

**4. Corregir tipos de datos (fechas, categoricos, numéricos).**

In [ ]:
df['PurchDate'] = pd.to_datetime(df['PurchDate'])

In [ ]:
cat_cols = [
    'Auction', 'Make', 'Model', 'Trim', 'SubModel',
    'Color', 'Transmission', 'WheelType',
    'Nationality', 'Size', 'TopThreeAmericanName',
    'PRIMEUNIT', 'AUCGUART', 'VNST'
]

In [ ]:
for col in cat_cols:
    df[col] = df[col].astype('category')

In [ ]:
num_cols = [
    'VehYear', 'VehicleAge', 'VehOdo',
    'MMRAcquisitionAuctionAveragePrice',
    'MMRAcquisitionAuctionCleanPrice',
    'MMRAcquisitionRetailAveragePrice',
    'MMRAcquisitonRetailCleanPrice',
    'MMRCurrentAuctionAveragePrice',
    'MMRCurrentAuctionCleanPrice',
    'MMRCurrentRetailAveragePrice',
    'MMRCurrentRetailCleanPrice',
    'VehBCost',
    'WarrantyCost'
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
id_cols = ['BYRNO', 'VNZIP1', 'WheelTypeID']

In [ ]:
for col in id_cols:
    df[col] = df[col].astype('category')

In [ ]:
binary_cols = ['IsBadBuy', 'IsOnlineSale']

for col in binary_cols:
    df[col] = df[col].astype(int)

In [ ]:
df.info()

*Corrección de tipos de datos:*

La variable PurchDate fue convertida a formato datetime para permitir análisis temporal.
Las variables categóricas fueron transformadas a tipo category para optimizar memoria y facilitar su posterior codificación.
Se verificó que las variables numéricas estén en formato adecuado (int o float).
Variables identificadoras (BYRNO, VNZIP1, WheelTypeID) fueron tratadas como categóricas.
Variables binarias (IsBadBuy, IsOnlineSale) se mantuvieron como numéricas enteras.

**5. Aplicar encoding a categoricas: OneHot, Ordinal, Target.**

In [ ]:
from sklearn.preprocessing import OneHotEncoder

onehot_cols = ['Transmission', 'Color', 'Auction', 'WheelType']

df = pd.get_dummies(df, columns=onehot_cols, drop_first=True)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_cols = ['Size']

encoder_ord = OrdinalEncoder()

df[ordinal_cols] = encoder_ord.fit_transform(df[ordinal_cols])

In [ ]:
!pip install category_encoders

In [ ]:
import category_encoders as ce

target_cols = ['Make', 'Model', 'SubModel', 'Trim', 'VNZIP1', 'BYRNO']

encoder_target = ce.TargetEncoder(cols=target_cols)

df[target_cols] = encoder_target.fit_transform(df[target_cols], df['IsBadBuy'])

In [ ]:
df.info()

*Encoding de variables categóricas:*

Se aplicó OneHot Encoding a variables con baja cardinalidad (Transmission, Color, Auction, WheelType).
Se utilizó Ordinal Encoding en variables con orden inherente (Size).
Para variables con alta cardinalidad (Make, Model, SubModel, etc.), se aplicó Target Encoding, utilizando la variable objetivo IsBadBuy.
Esta estrategia permite evitar la explosión de dimensiones y mejorar el rendimiento del modelo.

**6. Aplicar escalado: StandardScaler, MinMaxScaler,RobustScaler.**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# 1. Split
X = df.drop('IsBadBuy', axis=1)
y = df['IsBadBuy']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Escalado
scaler = RobustScaler()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

*Escalado de variables:*

Se evaluaron distintos métodos de escalado:
StandardScaler: estandariza a media 0 y desviación estándar 1.
MinMaxScaler: escala los valores en un rango entre 0 y 1.
RobustScaler: utiliza la mediana y el rango intercuartil, siendo robusto frente a outliers.
Dado que el dataset presenta valores extremos en variables como precios, se seleccionó RobustScaler como método principal.
El escalado se realizó después de dividir los datos en entrenamiento y prueba, ajustando el scaler únicamente con el conjunto de entrenamiento para evitar data leakage.